In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight

# Change working directory to the notebook's directory
notebook_dir = os.path.dirname(os.path.abspath("frozen_linear_sex.ipynb"))
os.chdir(notebook_dir)

# -------------------
# Config
# -------------------
n_folds = 5
train_pattern = "../../splits/hcp/train_subject_list_ag_{}"  # train_subject_list_ag_0 ... _4
val_pattern   = "../../splits/hcp/val_subject_list_ag_{}"    # val_subject_list_ag_0 ... _4
test_pattern  = "../../splits/hcp/test_subject_list_ag_{}"   # test_subject_list_ag_0 ... _4

meta_csv = "../../metadata/hcpagdev_metadata.csv"

feature_sets = ["../../latents/cls_hcpagdev_k8pcq4ai_300.npz",
                ]

out_dir = "cv_sex_results_agdev"
os.makedirs(out_dir, exist_ok=True)

# -------------------
# Load metadata (contains sex info)
# -------------------
df = pd.read_csv(meta_csv)
sex_map = dict(zip(df["src_subject_id"].astype(str), df["sex"].astype(str)))
# Map to binary: 1 = Female, 0 = Male (same mapping as your original)
sex_map = {k: (1 if v == "F" else 0) for k, v in sex_map.items()}

# -------------------
# Helpers for ID/key matching
# -------------------
def feature_key_variants(sid):
    """Yield possible keys to try for a split ID in features_dict."""
    yield sid
    if "_" in sid:
        yield sid.split("_")[0]
    if "/" in sid:
        yield sid.split("/")[0]
        if "_" in sid.split("/")[0]:
            yield sid.split("/")[0].split("_")[0]

def label_key_for(sid):
    """Return the key to lookup in sex_map (prefer prefix before underscore or before slash)."""
    if "/" in sid:
        sid = sid.split("/")[0]
    if "_" in sid:
        return sid.split("_")[0]
    return sid

def save_predictions_csv(csv_path, subject_ids, y_true, y_score, y_pred):
    df = pd.DataFrame({
        "subject_id": subject_ids,
        "y_true": y_true.astype(int),
        "y_score": y_score.astype(float),
        "y_pred": y_pred.astype(int),
    })
    os.makedirs(os.path.dirname(csv_path), exist_ok=True)
    df.to_csv(csv_path, index=False)


# -------------------
# Main loop: feature sets -> folds
# -------------------
rows = []  # accumulate per-fold results

for feat_file in feature_sets:
    features_dict = np.load(feat_file, allow_pickle=True)

    # prepare plotting figure (1 x n_folds)
    fig, axes = plt.subplots(1, n_folds, figsize=(4 * n_folds, 4), squeeze=False)
    axes = axes.ravel()

    for fold in range(n_folds):
        train_fname = train_pattern.format(fold)
        val_fname   = val_pattern.format(fold)
        test_fname  = test_pattern.format(fold)

        if not os.path.exists(train_fname) or not os.path.exists(val_fname):
            print(f"Warning: missing file(s) for fold {fold}: {train_fname} or {val_fname}. Skipping.")
            continue

        train_ids = np.loadtxt(train_fname, dtype=str)
        val_ids   = np.loadtxt(val_fname, dtype=str)
        test_ids  = np.loadtxt(test_fname, dtype=str)

        # Build lists of subject tuples (sid_in_split, feature_key, label_key)
        tr_ok = []
        for sid in train_ids:
            for key in feature_key_variants(sid):
                if key in features_dict:
                    lbl = label_key_for(sid)
                    if lbl in sex_map:
                        tr_ok.append((sid, key, lbl))
                        break

        va_ok = []
        for sid in val_ids:
            for key in feature_key_variants(sid):
                if key in features_dict:
                    lbl = label_key_for(sid)
                    if lbl in sex_map:
                        va_ok.append((sid, key, lbl))
                        break

        test_ok = []
        for sid in test_ids:
            for key in feature_key_variants(sid):
                if key in features_dict:
                    lbl = label_key_for(sid)
                    if  lbl in sex_map:
                        test_ok.append((sid, key, lbl))
                        break


        if len(tr_ok) == 0 or len(va_ok) == 0:
            print(f"Warning: fold {fold} for {feat_file} has empty train or val after filtering. Skipping fold.")
            continue

        # Build features and labels
        X_train = np.vstack([features_dict[key] for (_, key, _) in tr_ok])
        X_val   = np.vstack([features_dict[key] for (_, key, _) in va_ok])
        X_test  = np.vstack([features_dict[key] for (_, key, _) in test_ok])
        X_train = np.nan_to_num(X_train, nan=0.0)
        X_val   = np.nan_to_num(X_val, nan=0.0)
        X_test  = np.nan_to_num(X_test, nan=0.0)

        y_train = np.array([sex_map[lbl] for (_, _, lbl) in tr_ok], dtype=np.int64)
        y_val   = np.array([sex_map[lbl] for (_, _, lbl) in va_ok], dtype=np.int64)
        y_test = np.array([sex_map[lbl] for (_, _, lbl) in test_ok], dtype=np.int64)

        # Normalize features by training set
        X_mean = X_train.mean(axis=0, keepdims=True)
        X_std  = X_train.std(axis=0, keepdims=True) + 1e-8
        X_train_norm = (X_train - X_mean) / X_std
        X_val_norm   = (X_val - X_mean) / X_std
        X_test_norm  = (X_test - X_mean) / X_std

        # Train logistic regression
        clf = LogisticRegression(max_iter=2000, solver="lbfgs")
        clf.fit(X_train_norm, y_train)

        # Predictions and scores for validation
        y_pred = clf.predict(X_val_norm)
        if hasattr(clf, "predict_proba"):
            y_score = clf.predict_proba(X_val_norm)[:, 1]
        else:
            try:
                s = clf.decision_function(X_val_norm)
                y_score = (s - s.min()) / (s.max() - s.min() + 1e-8)
            except Exception:
                y_score = y_pred.astype(float)

        acc = accuracy_score(y_val, y_pred)
        try:
            auroc = roc_auc_score(y_val, y_score)
        except Exception:
            auroc = np.nan

        # Predictions and scores for test
        y_pred_test = clf.predict(X_test_norm)
        y_score_test = clf.predict_proba(X_test_norm)[:, 1]


        rows.append({
            "feature_file": os.path.basename(feat_file),
            "fold": int(fold),
            "n_train": int(len(y_train)),
            "n_val": int(len(y_val)),
            "Accuracy": float(acc),
            "AUROC": float(auroc) if not np.isnan(auroc) else np.nan
        })

        csv_path = f'CV_sex_results/frozen_linear_val_{fold}.csv'
        save_predictions_csv(csv_path,  [x[0] for x in va_ok], y_val, y_score, y_pred)
        csv_path = f'CV_sex_results/frozen_linear_test_{fold}.csv'
        save_predictions_csv(csv_path,  [x[0] for x in test_ok], y_test, y_score_test, y_pred_test)


        # Plot jittered GT vs Pred for visual check
        ax = axes[fold]
        jitter_gt = y_val + np.random.uniform(-0.05, 0.05, size=len(y_val))
        jitter_pred = y_pred + np.random.uniform(-0.05, 0.05, size=len(y_pred))
        ax.scatter(jitter_gt, jitter_pred, alpha=0.6)
        ax.plot([-.5, 1.5], [-.5, 1.5], 'r--')
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["Male", "Female"])
        ax.set_yticks([0, 1])
        ax.set_yticklabels(["Male", "Female"])
        ax.set_xlabel("Ground Truth Sex")
        ax.set_ylabel("Predicted Sex")
        ax.set_title(f"fold {fold}\nAcc={acc:.3f} AUROC={np.nan_to_num(auroc):.3f}")

    # end folds loop

# Save per-fold results and summary
df_res = pd.DataFrame(rows)
csv_out = os.path.join(out_dir, f"cv_sex_table_{os.path.basename(feat_file)}.csv")
df_res.to_csv(csv_out, index=False)
print(f"\nPer-fold results for {feat_file}:")
print(df_res)

print("Done. Per-fold CSVs and summaries saved to:", out_dir)





In [ ]:
# Aggregate df_res by feature_file
if 'df_res' in globals() and (isinstance(df_res, (pd.DataFrame)) and not df_res.empty):
    agg = df_res.groupby('feature_file')['Accuracy'].agg(['mean','std']).reset_index()
    agg['acc_mean_std'] = agg.apply(lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}", axis=1)
    agg = agg.rename(columns={'mean': 'rho_mean', 'std': 'rho_std'})[['feature_file','rho_mean','rho_std','acc_mean_std']]
    print('Mean ± std of rho_norm by feature_file:')
    display(agg[['feature_file','acc_mean_std']])
    os.makedirs(out_dir, exist_ok=True)
else:
    print('df_res not found or empty; run the previous cells to compute df_res before aggregating.')